# Voxtral HF Capability

> Capability implementation for Mistral Voxtral transcription through Hugging Face Transformers

In [ ]:
#| default_exp capability

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, Any, Optional, List, Union, ClassVar
import warnings

import torch
from fastcore.basics import patch

try:
    from transformers import VoxtralForConditionalGeneration, AutoProcessor
    VOXTRAL_AVAILABLE = True
except ImportError:
    VOXTRAL_AVAILABLE = False

# Stage 8 (Option C / PILLAR 1c): the tool re-bases onto ToolCapability (pure
# compute). The cache/persist bookends + the TranscriptionResult data noun moved
# OUT — the generic adapter (cjm-transcription-adapter-interface) owns the cache,
# and the result DTO lives in cjm-capability-primitives. No get_plugin_metadata.
from cjm_substrate.core.capability import ToolCapability, RELOAD_TRIGGER, EnvVarSpec
from cjm_capability_primitives.transcription import TranscriptionResult
from cjm_substrate.core.errors import (
    CapabilityInputError, CapabilityFatalError, CapabilityResourceError,
)
from cjm_substrate.utils.validation import (
    dict_to_config, config_to_dict, validate_config, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_MIN, SCHEMA_MAX, SCHEMA_ENUM
)

# Shared torch helpers (cjm-substrate-torch-utils): release + CUDA-OOM typing.
from cjm_substrate_torch_utils.memory import release_model
from cjm_substrate_torch_utils.oom import cuda_oom_to_capability_resource_error

# Shared HF helpers (cjm-substrate-hf-utils): cache-config mixin + progress download + OOM-typed load.
from cjm_substrate_hf_utils.cache_config import HFCacheConfig
from cjm_substrate_hf_utils.download import snapshot_download_with_progress
from cjm_substrate_hf_utils.loading import load_pretrained_with_oom

In [ ]:
#| export
@dataclass
class VoxtralHFCapabilityConfig(HFCacheConfig):
    """Configuration for Voxtral HF transcription capability."""
    model_id:str = field(
        default="mistralai/Voxtral-Mini-3B-2507",
        metadata={
            SCHEMA_TITLE: "Model ID",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Voxtral model to use. Mini is faster, Small is more accurate.",
            SCHEMA_ENUM: ["mistralai/Voxtral-Mini-3B-2507", "mistralai/Voxtral-Small-24B-2507"]
        }
    )
    device:str = field(
        default="auto",
        metadata={
            SCHEMA_TITLE: "Device",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Device for inference (auto will use CUDA if available)",
            SCHEMA_ENUM: ["auto", "cpu", "cuda"]
        }
    )
    dtype:str = field(
        default="auto",
        metadata={
            SCHEMA_TITLE: "Data Type",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Data type for model weights (auto uses bfloat16; set float32 explicitly for full precision)",
            SCHEMA_ENUM: ["auto", "bfloat16", "float16", "float32"]
        }
    )
    language:Optional[str] = field(
        default="en",
        metadata={
            SCHEMA_TITLE: "Language",
            SCHEMA_DESC: "Language code for transcription (e.g., 'en', 'es', 'fr')"
        }
    )
    max_new_tokens:int = field(
        default=25000,
        metadata={
            SCHEMA_TITLE: "Max New Tokens",
            SCHEMA_DESC: "Maximum number of tokens to generate",
            SCHEMA_MIN: 1,
            SCHEMA_MAX: 50000
        }
    )
    do_sample:bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Do Sample",
            SCHEMA_DESC: "Whether to use sampling (true) or greedy decoding (False)"
        }
    )
    temperature:float = field(
        default=1.0,
        metadata={
            SCHEMA_TITLE: "Temperature",
            SCHEMA_DESC: "Temperature for sampling (only used when do_sample=true)",
            SCHEMA_MIN: 0.0,
            SCHEMA_MAX: 2.0
        }
    )
    top_p:float = field(
        default=0.95,
        metadata={
            SCHEMA_TITLE: "Top P",
            SCHEMA_DESC: "Top-p (nucleus) sampling parameter (only used when do_sample=true)",
            SCHEMA_MIN: 0.0,
            SCHEMA_MAX: 1.0
        }
    )
    compile_model:bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Compile Model",
            SCHEMA_DESC: "Use torch.compile for potential speedup (requires PyTorch 2.0+)"
        }
    )
    load_in_8bit:bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Load in 8-bit",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Load model in 8-bit quantization (requires bitsandbytes)"
        }
    )
    load_in_4bit:bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Load in 4-bit",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Load model in 4-bit quantization (requires bitsandbytes)"
        }
    )


In [ ]:
#| export
class VoxtralHFCapability(ToolCapability):
    """Mistral Voxtral transcription capability via Hugging Face Transformers (stage 8: pure-compute tool capability).

    Native-surface model (PILLAR 1c): this tool is PURE COMPUTE — `transcribe`
    loads the model, runs inference, and builds the typed `TranscriptionResult`.
    The cache-check + persistence bookends + the per-call `force` control live in
    the generic transcription adapter (cjm-transcription-adapter-interface); the
    result DTO lives in cjm-capability-primitives; identity is derived from the
    installed distribution. No `get_plugin_metadata`, no `self.storage`."""

    # CR-4: declarative reload-triggers — substrate's reconfigure_with_triggers
    # walks this config_class's dataclass fields for RELOAD_TRIGGER metadata and
    # fires the corresponding `_release_<trigger>` method on field changes.
    config_class = VoxtralHFCapabilityConfig

    # Track 19 (CR-12 worker-env model): worker spawn env declared on the class.
    # CUDA_VISIBLE_DEVICES + OMP_NUM_THREADS are static; HF_HOME is templated to
    # the substrate models dir. The substrate resolves + injects at Popen.
    WORKER_ENV: ClassVar[List[EnvVarSpec]] = [
        EnvVarSpec(
            name="CUDA_VISIBLE_DEVICES",
            default="0",
            label="GPU Device",
            description="Which GPU index the worker uses.",
        ),
        EnvVarSpec(
            name="OMP_NUM_THREADS",
            default="4",
            label="OpenMP Threads",
            description="Thread cap for CPU-side ops.",
        ),
        EnvVarSpec(
            name="HF_HOME",
            default="${CJM_MODELS_DIR}/huggingface",
            label="HF Cache Directory",
            description="HuggingFace Hub cache root (templated to the substrate models dir).",
        ),
    ]

    def __init__(self):
        """Initialize the Voxtral HF capability with default configuration."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: VoxtralHFCapabilityConfig = None
        self.model = None
        self.processor = None
        self.device = None
        self.dtype = None

    @property
    def name(self) -> str: # Capability name identifier
        """Capability identity, derived from the installed distribution (PILLAR 1c).

        Runtime-derived: in the worker / in-env introspection `__package__`
        resolves; the manifest records the same value independently (the
        dual-mode generator reads it from the distribution)."""
        from importlib.metadata import metadata, packages_distributions
        dist = (packages_distributions().get(__package__) or [__package__.replace("_", "-")])[0]
        return metadata(dist)["Name"]

    @property
    def version(self) -> str: # Capability version string
        """Get the capability version string."""
        from cjm_capability_voxtral_hf import __version__
        return __version__

    def get_current_config(self) -> Dict[str, Any]: # Current configuration as dictionary
        """Return current configuration state."""
        if not self.config:
            return {}
        return config_to_dict(self.config)

    def get_config_schema(self) -> Dict[str, Any]: # JSON Schema for configuration
        """Return JSON Schema for UI generation."""
        return dataclass_to_jsonschema(VoxtralHFCapabilityConfig)

    @staticmethod
    def get_config_dataclass() -> VoxtralHFCapabilityConfig: # Configuration dataclass
        """Return dataclass describing the capability's configuration options."""
        return VoxtralHFCapabilityConfig

    def initialize(
        self,
        config: Optional[Any] = None # Configuration dataclass, dict, or None
    ) -> None:
        """First-time setup. CR-4: the manual model/device/dtype/quantization
        diff-and-reload is replaced by declarative RELOAD_TRIGGER metadata; the
        substrate's reconfigure path fires _release_model then re-applies config."""
        self._apply_config(config)
        self.logger.info(f"Initialized Voxtral HF capability with model '{self.config.model_id}' on device '{self.device}' with dtype '{self.dtype}'")

    def transcribe(
        self,
        audio: Union[str, Path], # Path to MODEL-READY audio (converted upstream)
        **kwargs # Provenance (source_start_time/source_end_time) stamped into metadata
    ) -> TranscriptionResult: # Typed transcription output
        """Transcribe model-ready audio using Voxtral — PURE COMPUTE.

        Stage 8 / PILLAR 1c: the cache-check + persistence bookends moved to the
        generic transcription adapter; this method loads the model, runs
        inference, and builds the typed result. Model params come from
        `self.config` (the CR-15 per-call override path is gone — the tool runs
        its effective config, no metadata lie); `source_start_time` /
        `source_end_time` ride the provenance kwarg channel into metadata."""
        # Validate + resolve the input path, then load the model.
        audio_path = self._prepare_audio(audio)
        self._load_model()

        # Effective config (no per-call override path).
        c = self.config
        model_id = c.model_id
        language = c.language
        max_new_tokens = c.max_new_tokens
        do_sample = c.do_sample
        temperature = c.temperature
        top_p = c.top_p

        # Prepare inputs
        self.logger.info(f"Processing audio with Voxtral {model_id}")

        inputs = self.processor.apply_transcription_request(
            language=language or "en",
            audio=str(audio_path),
            model_id=model_id
        )
        inputs = inputs.to(self.device, dtype=self.dtype)

        # Generation kwargs
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": do_sample,
        }
        # Add sampling parameters if sampling is enabled
        if do_sample:
            generation_kwargs["temperature"] = temperature
            generation_kwargs["top_p"] = top_p

        # Generate transcription. SG-47 Track B wraps the inference site so
        # CUDA OOM surfaces as CapabilityResourceError → CR-7 reactive-retry reloads.
        try:
            with torch.no_grad():
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    outputs = self.model.generate(
                        **inputs,
                        **generation_kwargs
                    )
        except torch.cuda.OutOfMemoryError as e:
            raise cuda_oom_to_capability_resource_error(
                e, label=f"Voxtral inference (model={model_id!r})",
            ) from e

        # Decode the output
        result_text = self.processor.batch_decode(
            outputs[:, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )[0]

        # Clean up tensors immediately
        del inputs
        del outputs
        if self.device == "cuda" and torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Capture provenance metadata passed via kwargs
        provenance_meta = {
            k: v for k, v in kwargs.items()
            if k in ['source_start_time', 'source_end_time']
        }

        # Create transcription result
        transcription_result = TranscriptionResult(
            text=result_text.strip(),
            confidence=None,  # Voxtral doesn't provide confidence scores
            segments=None,  # Voxtral doesn't provide segments by default
            metadata={
                "model": model_id,
                **provenance_meta,
                "language": language or "en",
                "device": self.device,
                "dtype": str(self.dtype),
            }
        )

        self.logger.info(f"Transcription completed: {len(result_text.split())} words")
        return transcription_result

In [ ]:
#| export
@patch
def _apply_config(
    self:VoxtralHFCapability,
    config: Optional[Any] = None # Configuration dataclass, dict, or None
) -> None:
    """CR-4: apply config + derive config-dependent state (device, dtype). No
    heavy-resource work. Called by initialize (first-time) and the substrate's
    reconfigure delta path. Model release on a model_id/device/dtype/quantization
    change is handled declaratively via RELOAD_TRIGGER -> _release_model."""
    self.config = dict_to_config(VoxtralHFCapabilityConfig, config or {})

    # Resolve device
    if self.config.device == "auto":
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        self.device = self.config.device

    # Resolve dtype. G9: `auto` resolves to bfloat16 UNIFORMLY (device-independent).
    # The prior auto->float32-on-CPU default doubled the CPU footprint — stage-3 G9:
    # Voxtral-Small-24B with auto on CPU hit ~83GB RSS + swap and was unusable, while
    # explicit bfloat16 on CPU ran fine. Full precision stays available via an
    # explicit dtype="float32".
    if self.config.dtype == "auto":
        self.dtype = torch.bfloat16
    else:
        dtype_map = {
            "bfloat16": torch.bfloat16,
            "float16": torch.float16,
            "float32": torch.float32
        }
        self.dtype = dtype_map[self.config.dtype]

In [ ]:
#| export
@patch
def _release_model(self:VoxtralHFCapability) -> None:
    """Unload the current model + processor and free GPU memory.

    Delegates to cjm-substrate-torch-utils' `release_model` (move-to-CPU / del / gc /
    empty_cache / synchronize) -- the single source of truth across torch GPU capabilitys."""
    if self.model is None and self.processor is None:
        return
    self.logger.info("Unloading Voxtral model for reconfiguration")
    release_model(self, ["model", "processor"], self.device or "cuda", logger=self.logger)


In [ ]:
#| export
@patch
def _load_model(self:VoxtralHFCapability) -> None:
    """Load the Voxtral model + processor (lazy).

    The heartbeat wraps BOTH the (potentially long, often quiet) snapshot download
    AND the silent from_pretrained build, so the substrate's prefetch stall detector
    always sees the (progress, message) tuple advance. snapshot_download_with_progress
    layers real per-file download % on top when the HF Hub tqdm callback fires.
    CUDA OOM on load surfaces as a typed CapabilityResourceError for CR-7 reactive retry."""
    if self.model is not None and self.processor is not None:
        return
    self.logger.info(f"Loading Voxtral model: {self.config.model_id}")

    # Built before the heartbeat block (instant). The snapshot below guarantees the
    # cache is populated, so the loads run local-only.
    model_kwargs = {
        "cache_dir": self.config.cache_dir,
        "revision": self.config.revision,
        "trust_remote_code": self.config.trust_remote_code,
        "local_files_only": True,
        "device_map": self.device,
    }
    if self.config.load_in_8bit:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif self.config.load_in_4bit:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)
    else:
        model_kwargs["dtype"] = self.dtype

    try:
        with self.heartbeat(f"loading Voxtral model {self.config.model_id}"):
            # Download (honors air-gap via local_files_only). The heartbeat is the
            # floor here; the tqdm hook adds real "downloading <file>" % when it fires.
            # G6: skip the Mistral original-format `consolidated.safetensors` — on
            # Voxtral-Small-24B it is a 48.5GB second full copy of the weights (Mini
            # ships an 8.75GB one) that HF `from_pretrained` never loads (it reads the
            # sharded `model-*.safetensors` via `model.safetensors.index.json`). The
            # `ignore_patterns` rides snapshot_download's **kwargs passthrough (no
            # shared-util change) and no-ops on repos without a consolidated copy.
            snapshot_download_with_progress(
                self.config.model_id,
                report_progress=self.report_progress,
                cache_dir=self.config.cache_dir,
                revision=self.config.revision,
                local_files_only=self.config.local_files_only,
                ignore_patterns=["consolidated*"],
            )
            self.processor = AutoProcessor.from_pretrained(
                self.config.model_id,
                cache_dir=self.config.cache_dir,
                revision=self.config.revision,
                trust_remote_code=self.config.trust_remote_code,
                local_files_only=True,
            )
            self.model = load_pretrained_with_oom(
                VoxtralForConditionalGeneration,
                self.config.model_id,
                label=f"loading Voxtral model {self.config.model_id!r}",
                **model_kwargs,
            )

        if self.config.compile_model and hasattr(torch, "compile"):
            self.model = torch.compile(self.model)
            self.logger.info("Model compiled with torch.compile")
        self.logger.info("Voxtral model loaded successfully")
    except CapabilityResourceError:
        raise  # already typed by load_pretrained_with_oom
    except torch.cuda.OutOfMemoryError as e:
        # Defensive: OOM outside the wrapped model load (processor / compile).
        raise cuda_oom_to_capability_resource_error(
            e, label=f"loading Voxtral model {self.config.model_id!r}",
        ) from e
    except Exception as e:
        raise CapabilityFatalError(f"Failed to load Voxtral model: {e}") from e

In [ ]:
#| export
@patch
def _prepare_audio(
    self:VoxtralHFCapability,
    audio: Union[str, Path] # Path to a decodable audio file
) -> str: # The audio file path
    """Validate the audio input and return it as a path string.

    The caller (orchestration / proxy) guarantees a model-ready audio file;
    in-memory preparation is no longer a capability responsibility."""
    if isinstance(audio, (str, Path)):
        return str(audio)
    raise CapabilityInputError(  # SG-47: typed input-validation (multi-inherits ValueError)
        f"Unsupported audio input type: {type(audio)}; expected a file path (str or Path)",
        fields_invalid=["audio"],
    )


In [ ]:
#| export
@patch
def is_available(self:VoxtralHFCapability) -> bool: # True if Voxtral and its dependencies are available
    """Check if Voxtral is available."""
    return VOXTRAL_AVAILABLE


In [ ]:
#| export
@patch
def prefetch(self:VoxtralHFCapability) -> None:
    """CR-4 (SG-19): eagerly load the model + processor so the first execute()
    doesn't pay the download/load cost. Idempotent via _load_model's None-guard."""
    self._load_model()


In [ ]:
#| export
@patch
def on_disable(self:VoxtralHFCapability) -> None:
    """CR-2: release the GPU model + processor when the operator disables the
    capability (the worker stays alive); lazy reload on the next execute."""
    self._release_model()


In [ ]:
#| export
@patch
def cleanup(self:VoxtralHFCapability) -> None:
    """Release the model + processor (CR-4: delegates to `_release_model`)."""
    self._release_model()


## Testing the Capability

In [ ]:
# Basic functionality (stage-8: pure-compute ToolCapability).
from cjm_substrate.core.capability import ToolCapability

capability = VoxtralHFCapability()
assert isinstance(capability, ToolCapability)
print(f"Voxtral available: {capability.is_available()}")
print(f"Capability version: {capability.version}")
print(f"Config class: {capability.config_class.__name__}")

# Native-surface model: pure-compute `transcribe` replaces the fused `execute`;
# cache/persist + identity moved out. `name` is derived from the installed dist
# at runtime (worker / in-env introspection), so it is NOT exercised in-notebook
# where __package__ is unset.
assert hasattr(capability, "transcribe") and not hasattr(capability, "execute")
assert not hasattr(capability, "supported_formats")

# Streaming override retired (no current consumer; descoped GUI host) — re-add as
# a native method under a first-principles streaming design. The base
# ToolCapability still provides a default execute_stream, so we assert the
# OVERRIDE is gone from the class __dict__, not hasattr (which sees the base).
assert "execute_stream" not in VoxtralHFCapability.__dict__
assert "supports_streaming" not in VoxtralHFCapability.__dict__
print("VoxtralHFCapability is a pure-compute ToolCapability (transcribe; no execute/cache/storage/streaming-override)")

In [ ]:
# Test configuration dataclass
from dataclasses import fields

print("Available models:")
model_field = next(f for f in fields(VoxtralHFCapabilityConfig) if f.name == "model_id")
for model in model_field.metadata.get(SCHEMA_ENUM, []):
    print(f"  - {model}")

Available models:
  - mistralai/Voxtral-Mini-3B-2507
  - mistralai/Voxtral-Small-24B-2507


In [ ]:
# Test configuration validation
test_configs = [
    ({"model_id": "mistralai/Voxtral-Mini-3B-2507"}, "Valid config"),
    ({"model_id": "invalid_model"}, "Invalid model"),
    ({"model_id": "mistralai/Voxtral-Mini-3B-2507", "temperature": 2.5}, "Temperature out of range"),
]

for config, description in test_configs:
    try:
        test_cfg = dict_to_config(VoxtralHFCapabilityConfig, config, validate=True)
        print(f"{description}: Valid=True")
    except ValueError as e:
        print(f"{description}: Valid=False")
        print(f"  Error: {str(e)[:100]}")

Valid config: Valid=True
Invalid model: Valid=False
  Error: model_id: 'invalid_model' is not one of ['mistralai/Voxtral-Mini-3B-2507', 'mistralai/Voxtral-Small-
Temperature out of range: Valid=False
  Error: temperature: 2.5 is greater than maximum 2.0


In [ ]:
# Test initialization and get_current_config (returns dict now)
capability.initialize({"model_id": "mistralai/Voxtral-Mini-3B-2507", "device": "cpu"})
current_config = capability.get_current_config()
print(f"Current config (dict): model_id={current_config['model_id']}")

Current config (dict): model_id=mistralai/Voxtral-Mini-3B-2507


In [ ]:
#| eval: false
# Test get_config_schema for UI generation
import json

schema = capability.get_config_schema()
print("JSON Schema for VoxtralHFCapabilityConfig:")
print(f"  Name: {schema['name']}")
print(f"  Properties count: {len(schema['properties'])}")
print(f"  Model field enum: {schema['properties']['model_id'].get('enum', [])}")
print(f"\nSample properties:")
print(json.dumps({k: v for k, v in list(schema['properties'].items())[:3]}, indent=2))

JSON Schema for VoxtralHFCapabilityConfig:
  Name: VoxtralHFCapabilityConfig
  Properties count: 14
  Model field enum: ['mistralai/Voxtral-Mini-3B-2507', 'mistralai/Voxtral-Small-24B-2507']

Sample properties:
{
  "model_id": {
    "type": "string",
    "title": "Model ID",
    "description": "Voxtral model to use. Mini is faster, Small is more accurate.",
    "enum": [
      "mistralai/Voxtral-Mini-3B-2507",
      "mistralai/Voxtral-Small-24B-2507"
    ],
    "default": "mistralai/Voxtral-Mini-3B-2507"
  },
  "device": {
    "type": "string",
    "title": "Device",
    "description": "Device for inference (auto will use CUDA if available)",
    "enum": [
      "auto",
      "cpu",
      "cuda"
    ],
    "default": "auto"
  },
  "dtype": {
    "type": "string",
    "title": "Data Type",
    "description": "Data type for model weights (auto will use bfloat16 on GPU, float32 on CPU)",
    "enum": [
      "auto",
      "bfloat16",
      "float16",
      "float32"
    ],
    "default": "

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()